# Univariate Analysis
In this notebook, we analyse the final dataset from `01_Data_Cleaner.ipynb`, using the decisions in `Data_exploration.ipynb` and `DS_Comments.md`.

We start from `SBA_clean.csv`, generated by the cleaner, and examine each retained variable in turn.

## Variables retained after cleaning
We analyse all 17 final columns. In the table below, we follow the cleaner's column order and indicate the section where we examine each variable.

| Variable | Meaning after cleaning | Analysis | Section |
|---|---|---|---|
| `LocationID` | Lender identifier, stored as a string | Frequencies, top 20 and cumulative concentration | 7 |
| `BorrState` | Borrower state, with selected territories grouped | Full frequency table and top-15 bar chart | 6 |
| `LenderType` | Bank / Credit union / Non-bank, derived from registry IDs | Counts, percentages and bar chart | 3 |
| `GrossApproval` | Approved loan amount in USD | Summary, histogram, log view, box plot and IQR flags | 2, 2a, 2d |
| `GuaranteedPct` | SBA-guaranteed amount divided by approved amount | Summary, coverage-level bar chart and IQR flags | 2, 2c, 2d |
| `ApprovalFY` | Fiscal year of approval | Ordered counts, percentages and bar chart | 5 |
| `ApprovalFQ` | Fiscal quarter derived from approval date | Ordered counts, percentages and bar chart | 5 |
| `ChannelGroup` | Grouped processing method | Counts, percentages and bar chart | 3 |
| `InitialInterestRate` | Initial rate in percent, after retaining positive rates | Summary, histogram, box plot and IQR flags | 2, 2b, 2d |
| `FixedorVariableInterestInd` | Fixed / variable rate category | Counts, percentages and bar chart | 3 |
| `Sector` | Sector derived from the first two NAICS digits | Counts, percentages and bar chart | 3 |
| `IsFranchise` | Flag for a non-missing franchise code | Counts and percentages for 0/1, bar chart | 4 |
| `BusinessType` | Legal form, with missing values filled as CORPORATION | Counts, percentages and bar chart | 3 |
| `BusinessAgeNew` | Grouped business age, including Unknown | Counts, percentages and bar chart | 3 |
| `IsRevolver` | Flag for RevolverStatus equal to Y | Counts and percentages for 0/1, bar chart | 4 |
| `HasCollateral` | Flag for CollateralInd equal to Y | Counts and percentages for 0/1, bar chart | 4 |
| `Default` | Charge-off within seven years of approval | Counts, percentages and class-balance chart | 1 |

We cover 3 numeric measurements, 6 financing/borrower categories, 3 binary features, 2 temporal categories, 1 geographic category, 1 lender identifier and 1 binary target.

We count **loans** in every frequency table. We treat `LocationID` as a label and years, quarters and binary flags as categories. We focus on individual distributions, leaving relationships with `Default`, correlations and comparisons between variables to bivariate analysis.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display

# LocationID is a label: preserve leading zeros.
df = pd.read_csv("SBA_clean.csv", dtype={"LocationID": str}, low_memory=False)
df.shape

## Input check
We start from the cleaner output and check that it contains the expected columns, valid encodings and no missing values. Our analysis uses the retained variables, following the exclusions already made during cleaning. Counts may differ from the initial exploration because cleaning also filters rows.

We use the cleaning guide's 478,956 rows and 17 columns as reference values for the same source snapshot. We check the actual input below.

In [ ]:
numeric = ["GrossApproval", "InitialInterestRate", "GuaranteedPct"]
categorical = [
    "ChannelGroup", "LenderType", "FixedorVariableInterestInd",
    "Sector", "BusinessType", "BusinessAgeNew",
]
binary = ["IsFranchise", "IsRevolver", "HasCollateral"]
expected = numeric + categorical + binary + [
    "Default", "ApprovalFY", "ApprovalFQ", "BorrState", "LocationID",
]

assert set(df.columns) == set(expected), "Use the final 17 columns from 01_Data_Cleaner.ipynb."
assert not df.empty, "The clean dataset is empty."
assert not df.isna().any().any(), "Missing values found: check the cleaner output."
for col in binary + ["Default"]:
    assert df[col].isin([0, 1]).all(), f"Unexpected values in {col}."
assert df["GrossApproval"].gt(0).all(), "GrossApproval must be positive for the log view."
assert df["InitialInterestRate"].gt(0).all(), "The cleaner removes non-positive rates."
assert df["GuaranteedPct"].between(0, 1).all(), "Check the guarantee ratio."
assert df["ApprovalFY"].isin(range(2010, 2020)).all(), "Check the fiscal-year range."
assert df["ApprovalFQ"].isin([1, 2, 3, 4]).all(), "Check fiscal-quarter encoding."

pd.DataFrame({
    "dtype": df.dtypes.astype(str),
    "n_unique": df.nunique(),
    "missing": df.isna().sum(),
})

## Frequency tables and charts
We define a common function to calculate counts and percentages and draw a bar chart for each categorical variable. We then analyse each variable in its own cell and store the full-precision tables in `frequency_tables`.

In [ ]:
def analyse_category(data, column, order=None, horizontal=True):
    """Display counts and percentages, plot the distribution and return its table."""
    counts = data[column].value_counts()
    if order is not None:
        unexpected = counts.index.difference(order)
        if len(unexpected):
            raise ValueError(f"Unexpected categories in {column}: {list(unexpected)}")
        counts = counts.reindex(order, fill_value=0)

    table = counts.rename("count").to_frame()
    table.index.name = column
    table["pct"] = table["count"] / len(data) * 100
    display(table.round(2))

    if horizontal:
        ax = table["pct"].iloc[::-1].plot.barh(
            figsize=(9, max(3, len(table) * .3)), title=column,
        )
        ax.set_xlabel("Loans (%)")
    else:
        ax = table["pct"].plot.bar(figsize=(9, 4), rot=0, title=column)
        ax.set_ylabel("Loans (%)")
    plt.tight_layout()
    plt.show()
    return table


frequency_tables = {}

Here we created the table we're using for the next points 1 and 2.

## 1. Target: Default
We first examine the target distribution to assess class imbalance before any resampling. We report both counts and percentages.

We interpret `1` as a charge-off within seven years of approval and `0` as no charge-off within that window. A value of `0` does not necessarily mean the loan was paid in full. This first step describes the target's prevalence; identifying its predictors requires further analysis.

In [ ]:
target = analyse_category(df, "Default", order=[0, 1], horizontal=False)
frequency_tables["Default"] = target

## 2. Numeric summary
We summarise the three numeric variables using the mean, standard deviation, median, quartiles and upper percentiles. We compare the mean and median, alongside skewness and the graphs, to assess asymmetry.

We keep the units in mind: `GrossApproval` is in USD, `InitialInterestRate` is already expressed in percent (6.5 means 6.5%), and `GuaranteedPct` is a fraction (0.75 means 75%).

In [ ]:
summary = df[numeric].describe(percentiles=[.01, .25, .50, .75, .95, .99]).T
summary["IQR"] = summary["75%"] - summary["25%"]
summary["skewness"] = df[numeric].skew()
summary.round(4)

### 2a. GrossApproval
We examine loan size using a histogram and a box plot. Since the initial exploration identified a long right tail, we also plot `log10(GrossApproval)` to make the distribution easier to read.

In the log view, we interpret 4 as USD 10,000, 5 as USD 100,000 and 6 as USD 1,000,000.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].hist(df["GrossApproval"], bins=50)
axes[0].set_xlabel("GrossApproval (USD)")
axes[0].set_ylabel("Loans")
axes[1].hist(np.log10(df["GrossApproval"]), bins=40)
axes[1].set_xlabel("log10(GrossApproval in USD)")
axes[1].set_ylabel("Loans")
axes[2].boxplot(df["GrossApproval"], vert=False)
axes[2].set_xlabel("GrossApproval (USD)")
fig.suptitle("Loan size")
plt.tight_layout()
plt.show()

### 2b. InitialInterestRate
We examine the typical interest rate, its spread and its upper tail. The cleaner has already removed non-positive rates. We inspect unusually high values as potential extremes while keeping in mind that they may be valid observations.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(df["InitialInterestRate"], bins=40)
axes[0].set_xlabel("Initial interest rate (%)")
axes[0].set_ylabel("Loans")
axes[1].boxplot(df["InitialInterestRate"], vert=False)
axes[1].set_xlabel("Initial interest rate (%)")
fig.suptitle("InitialInterestRate")
plt.tight_layout()
plt.show()

### 2c. GuaranteedPct
We examine which SBA guarantee levels are most common. We express coverage as a percentage and round it to two decimal places for the frequency table, retaining the original ratio for the numeric summary.

We plot the four most frequent coverage levels and combine the remaining levels as `Other coverage levels`. Each bar shows the percentage of all loans, with its count alongside it. The subtitle shows the combined share of the top levels.

We distinguish the two percentages: a bar labelled `75% coverage` refers to the share of each loan guaranteed by the SBA, while its length shows the share of loans with that coverage. `Other coverage levels` combines several levels and is not a single guarantee percentage.

In [ ]:
coverage_pct = df["GuaranteedPct"] * 100
coverage_levels = coverage_pct.round(2).value_counts().rename("count").to_frame()
coverage_levels.index.name = "coverage_pct"
coverage_levels["pct"] = coverage_levels["count"] / len(df) * 100

# Keep the main levels separate and combine the remaining loans for display.
top_coverage = coverage_levels.head(4).copy()
top_share = top_coverage["pct"].sum()
coverage_chart = top_coverage.copy()
coverage_chart.index = [f"{level:g}% coverage" for level in coverage_chart.index]
remaining = int(coverage_levels["count"].iloc[4:].sum())
if remaining > 0:
    coverage_chart.loc["Other coverage levels"] = [remaining, remaining / len(df) * 100]
display(coverage_chart.round(2))

fig, ax = plt.subplots(figsize=(11, 6))
colors = ["#2563eb" if label != "Other coverage levels" else "#94a3b8"
          for label in coverage_chart.index]
bars = ax.barh(coverage_chart.index, coverage_chart["pct"], color=colors)
ax.invert_yaxis()
ax.bar_label(
    bars,
    labels=[f"{row.pct:.2f}% ({int(row.count):,} loans)"
            for row in coverage_chart.itertuples()],
    padding=6, fontsize=9,
)
ax.set_xlim(0, coverage_chart["pct"].max() * 1.5)
ax.set_xlabel("Share of all loans (%)")
ax.set_ylabel("Share of each loan guaranteed by the SBA")
ax.set_title(
    f"SBA guarantee coverage: most frequent levels\n"
    f"Top {len(top_coverage)} levels account for {top_share:.2f}% of all loans",
    loc="left", pad=14,
)
ax.xaxis.grid(True, alpha=.2)
ax.set_axisbelow(True)
ax.spines[["top", "right"]].set_visible(False)
fig.text(.02, .02, "Labels show the loan share and count. Coverage levels are rounded to 2 decimal places.", fontsize=9)
plt.tight_layout(rect=[0, .05, 1, 1])
plt.show()

### 2d. Extreme values: descriptive check
We count observations outside the 1.5 × IQR fences for each numeric variable. We use these flags to describe extremes while retaining all observations: valid large loans and common guarantee levels can fall outside the fences, especially when distributions are skewed or concentrated.

In [ ]:
q1 = df[numeric].quantile(.25)
q3 = df[numeric].quantile(.75)
iqr = q3 - q1
lower = q1 - 1.5 * iqr
upper = q3 + 1.5 * iqr
outside = df[numeric].lt(lower) | df[numeric].gt(upper)

pd.DataFrame({
    "lower_fence": lower,
    "upper_fence": upper,
    "outside_count": outside.sum(),
    "outside_pct": outside.mean() * 100,
}).round(4)

## 3. Financing and borrower categories
We analyse each variable separately through a frequency table and a percentage chart to describe the financing and borrower composition of our sample.

### 3a. ChannelGroup
We examine how loans are distributed across the six grouped processing channels.

In [ ]:
frequency_tables["ChannelGroup"] = analyse_category(df, "ChannelGroup")

### 3b. LenderType
We calculate the share of loans issued by each lender type. We follow the cleaner's classification, which gives Bank priority when both registry IDs are present.

In [ ]:
frequency_tables["LenderType"] = analyse_category(df, "LenderType")

### 3c. FixedorVariableInterestInd
We compare the proportions of loans with fixed (`F`) and variable (`V`) interest rates.

In [ ]:
frequency_tables["FixedorVariableInterestInd"] = analyse_category(df, "FixedorVariableInterestInd")

### 3d. Sector
We examine the distribution of loans across grouped sectors. When interpreting `Other Services`, we account for the cleaner's grouping of code 81 together with missing or unmapped NAICS values.

In [ ]:
frequency_tables["Sector"] = analyse_category(df, "Sector")

### 3e. BusinessType
We describe the distribution of borrowers' legal forms. We account for the cleaner's replacement of missing source values with `CORPORATION`.

In [ ]:
frequency_tables["BusinessType"] = analyse_category(df, "BusinessType")

### 3f. BusinessAgeNew
We examine the grouped business-age categories and keep `Unknown` visible. We interpret `Existing (2y+)` according to the cleaner's definition, which also includes change-of-ownership records.

In [ ]:
frequency_tables["BusinessAgeNew"] = analyse_category(
    df, "BusinessAgeNew", order=["Startup", "New (<2y)", "Existing (2y+)", "Unknown"],
)

## 4. Binary features
We examine the prevalence of franchise status, revolving credit and reported collateral. We show both 0 and 1 as categories, even though these flags are stored as numbers.

We interpret `0` as the cleaner's condition for `1` not being met, using the definitions of the original encoding.

### 4a. IsFranchise
We calculate the proportion of loans with a franchise code present (`1`).

In [ ]:
frequency_tables["IsFranchise"] = analyse_category(df, "IsFranchise", order=[0, 1], horizontal=False)

### 4b. IsRevolver
We calculate the proportion of loans reported as revolving credit (`1`, derived from `RevolverStatus = Y`).

In [ ]:
frequency_tables["IsRevolver"] = analyse_category(df, "IsRevolver", order=[0, 1], horizontal=False)

### 4c. HasCollateral
We calculate the proportion of loans with reported collateral (`1`, derived from `CollateralInd = Y`).

In [ ]:
frequency_tables["HasCollateral"] = analyse_category(df, "HasCollateral", order=[0, 1], horizontal=False)

## 5. Fiscal year and quarter
We treat `ApprovalFY` and `ApprovalFQ` as ordered discrete categories to examine how our sample is distributed over time.

We use the fiscal-quarter definition from cleaning: Q1 = Oct–Dec, Q2 = Jan–Mar, Q3 = Apr–Jun and Q4 = Jul–Sep. We pool quarter counts across all fiscal years, so we interpret this distribution as sample composition rather than evidence of a seasonal mechanism.

### 5a. ApprovalFY
We examine the number and percentage of loans approved in each fiscal year, keeping the years in chronological order.

In [ ]:
frequency_tables["ApprovalFY"] = analyse_category(
    df, "ApprovalFY", order=range(2010, 2020), horizontal=False,
)

### 5b. ApprovalFQ
We examine the number and percentage of loans approved in each fiscal quarter, ordered from Q1 to Q4.

In [ ]:
frequency_tables["ApprovalFQ"] = analyse_category(
    df, "ApprovalFQ", order=[1, 2, 3, 4], horizontal=False,
)

## 6. BorrState
We examine geographic concentration using the complete frequency table, including `Territories`. For readability, we plot the top 15 categories and calculate their combined share.

We calculate all percentages using **all cleaned loans** as the denominator, including states omitted from the chart.

In [ ]:
states = df["BorrState"].value_counts().rename("count").to_frame()
states["pct"] = states["count"] / len(df) * 100
frequency_tables["BorrState"] = states
display(states.round(2))
print(f"Top-15 state categories: {states['pct'].head(15).sum():.2f}% of loans")

ax = states["pct"].head(15).iloc[::-1].plot.barh(figsize=(9, 5), title="BorrState: top 15")
ax.set_xlabel("All loans (%)")
plt.tight_layout()
plt.show()

## 7. LocationID: lender concentration
We treat `LocationID` as a lender label and examine the frequency of each ID. We revisit the concentration observed in the initial exploration using the cleaned sample.

We calculate the number of distinct IDs, the top-20 share and how many IDs account for 50%, 80% and 90% of loans. For the display table, we group the remaining IDs as `Other`. We interpret this category as an aggregate of lenders and retain the original IDs in the dataset.

In [ ]:
lenders = df["LocationID"].value_counts().rename("count").to_frame()
lenders["pct"] = lenders["count"] / len(df) * 100
lenders["cum_pct"] = lenders["count"].cumsum() / len(df) * 100
lenders["rank"] = np.arange(1, len(lenders) + 1)
frequency_tables["LocationID"] = lenders

print(f"Distinct lender IDs: {len(lenders):,}")
print(f"Top-20 share: {lenders['pct'].head(20).sum():.2f}% of loans")
concentration = pd.DataFrame({
    "loan_share_pct": [50, 80, 90],
    "lender_IDs_needed": [int(lenders.loc[lenders["cum_pct"] >= t, "rank"].iloc[0])
                          for t in [50, 80, 90]],
})
display(concentration)

top_lenders = lenders["count"].head(20).copy()
if len(lenders) > 20:
    top_lenders.loc["Other"] = lenders["count"].iloc[20:].sum()
top_lenders = top_lenders.to_frame()
top_lenders["pct"] = top_lenders["count"] / len(df) * 100
display(top_lenders.round(2))

fig, axes = plt.subplots(1, 2, figsize=(14, 7))
top_lenders["pct"].iloc[::-1].plot.barh(ax=axes[0], title="Top 20 lender IDs + Other")
axes[0].set_xlabel("All loans (%)")
axes[1].plot(lenders["rank"], lenders["cum_pct"])
axes[1].set_xlabel("Number of lender IDs, ranked by loan count")
axes[1].set_ylabel("Cumulative loans (%)")
axes[1].set_title("Lender concentration")
plt.tight_layout()
plt.show()

## 8. How we interpret the outputs
We use the tables and charts above to describe our sample:

- **Frequency tables:** we read `count` as the number of loans and `pct` as their percentage of all cleaned loans. We identify the most common and least represented categories.
- **Numeric summaries:** we use `50%` as the median and `25%`–`75%` as the interval containing the central half of observations. We examine its width through the IQR, compare the mean and median with the histogram, and inspect values flagged by the IQR fences.
- **Target and binary features:** we use the percentage for value 1 to describe the prevalence of each recorded condition. For `Default`, we refer specifically to charge-off within seven years.
- **Geography and lenders:** we interpret top-category percentages relative to all loans and use cumulative lender shares to assess concentration.

We base our conclusions on the outputs obtained after running the notebook on `SBA_clean.csv`. Here we describe each variable individually; we leave associations with default for a separate analysis.